# Exploración de Datos — CNN+LSTM Biomecánica
**Proyecto:** Reconocimiento de Ejercicios  
**Equipo:** Christopher Zúñiga (C28730) · Adrian Arrieta Orozco (B70734)

In [ ]:
import cv2, mediapipe as mp, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

## Extracción parcial — prueba con 1 video

In [ ]:
# Define the video and label
video_path = "dataset_limpio/deadlift/deadlift_01.mp4"
video_name = "deadlift_01.mp4"
label = "deadlift"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: Could not open video file.")
else:
    print("Video file opened successfully!")
# Get video properties
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Total frames: {frame_count}, FPS: {fps}")
 
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
    running_mode=VisionRunningMode.IMAGE)
# List to collect all our frame data
dataset_rows = []
with PoseLandmarker.create_from_options(options) as landmarker:
    frame_index = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert frame to MediaPipe Image object
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Apply detect() to each frame
        pose_landmarker_result = landmarker.detect(mp_image)
        
        # Check if any landmarks were detected
        if pose_landmarker_result.pose_landmarks:
            # We assume there's one person per frame, so we take index [0]
            landmarks = pose_landmarker_result.pose_landmarks[0]
            
            # Basic metadata for the frame
            row_data = {
                'video_name': video_name,
                'label': label,
                'frame_index': frame_index
            }
            
            # MediaPipe Pose has 33 landmarks. We'll flatten them into the row.
            for i, landmark in enumerate(landmarks):
                row_data[f'x{i}'] = landmark.x
                row_data[f'y{i}'] = landmark.y
                row_data[f'z{i}'] = landmark.z
                row_data[f'vis{i}'] = landmark.visibility
                
            dataset_rows.append(row_data)
            
        frame_index += 1
        
    cap.release()
# Convert the collected data into a DataFrame
df = pd.DataFrame(dataset_rows)
# Save to a CSV dataset file
output_csv = "pose_landmarks_dataset.csv"
df.to_csv(output_csv, index=False)
print(f"Extraction complete! Saved {len(df)} frames to '{output_csv}'.")
# Preview the first few rows of the generated dataset
df.head()

## Extracción completa — todos los videos del dataset

In [ ]:
from pathlib import Path
import cv2, mediapipe as mp, numpy as np, pandas as pd

# Each tuple: (folder_path, exercise_name, form_label)
CLASSES = [
    ("dataset_limpio/deadlift",     "deadlift", "correct"),
    ("dataset_limpio/deadlift_bad", "deadlift", "incorrect"),
    ("dataset_limpio/pull Up",      "pull_up",  "correct"),
    ("dataset_limpio/pull_up_bad",  "pull_up",  "incorrect"),
    ("dataset_limpio/squat",        "squat",    "correct"),
    ("dataset_limpio/squat_bad",    "squat",    "incorrect"),
]

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}

BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="pose_landmarker_full.task"),
    running_mode=VisionRunningMode.IMAGE,
)

all_rows = []

with PoseLandmarker.create_from_options(options) as landmarker:
    for folder, exercise, form in CLASSES:
        folder_path = Path(folder)
        videos = sorted(
            [f for f in folder_path.iterdir() if f.suffix.lower() in VIDEO_EXTENSIONS],
            key=lambda p: p.name.lower(),
        )
        print(f"\n▶ {folder}  ({len(videos)} videos)  label={form}")

        for vid in videos:
            cap = cv2.VideoCapture(str(vid))
            if not cap.isOpened():
                print(f"  ✗ Cannot open: {vid.name}")
                continue

            frame_index     = 0
            frames_detected = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result   = landmarker.detect(mp_image)

                if result.pose_landmarks:
                    lms = result.pose_landmarks[0]
                    row = {
                        "video_name":  vid.name,
                        "exercise":    exercise,
                        "label":       form,          # "correct" or "incorrect"
                        "frame_index": frame_index,
                    }
                    for i, lm in enumerate(lms):
                        row[f"x{i}"]   = lm.x
                        row[f"y{i}"]   = lm.y
                        row[f"z{i}"]   = lm.z
                        row[f"vis{i}"] = lm.visibility
                    all_rows.append(row)
                    frames_detected += 1

                frame_index += 1

            cap.release()
            print(f"  ✓ {vid.name:40s}  {frames_detected}/{frame_index} frames detected")

df_full = pd.DataFrame(all_rows)
df_full.to_csv("pose_landmarks_dataset.csv", index=False)

print(f"\nDone! {len(df_full)} rows → 'pose_landmarks_dataset.csv'")
print(f"Shape: {df_full.shape}")
print(df_full.groupby(["exercise", "label"]).size().to_string())


## Revisar poses faltantes o nulas en dataset

In [15]:
  df = pd.read_csv("pose_landmarks_dataset.csv")
  print(df.shape)
  print(df.head())
  print(df.dtypes)

(3617, 136)
        video_name  exercise    label  frame_index        x0        y0  \
0  deadlift_01.mp4  deadlift  correct            0  0.682767  0.480350   
1  deadlift_01.mp4  deadlift  correct            1  0.683238  0.477310   
2  deadlift_01.mp4  deadlift  correct            2  0.683499  0.476626   
3  deadlift_01.mp4  deadlift  correct            3  0.682195  0.452999   
4  deadlift_01.mp4  deadlift  correct            4  0.682466  0.441791   

         z0      vis0        x1        y1  ...       z30     vis30       x31  \
0 -0.660312  0.999969  0.690322  0.461134  ... -0.101245  0.767949  0.736400   
1 -0.723228  0.999967  0.690567  0.457754  ... -0.099390  0.757113  0.734471   
2 -0.710239  0.999971  0.691103  0.456390  ... -0.101834  0.764345  0.734195   
3 -0.702665  0.999981  0.689915  0.433839  ... -0.116103  0.845333  0.736953   
4 -0.694467  0.999992  0.690352  0.423060  ... -0.087057  0.847425  0.734845   

        y31       z31     vis31       x32       y32       z32 

In [ ]:
# Step 3 — Missing data analysis

In [ ]:
# Step 4 — Histograms: landmark coordinate distributions

In [ ]:
# Step 5 — Heatmaps: landmark correlations

In [ ]:
# Step 6 — Joint angle distributions per exercise